# 03 — ML combination vs linear benchmark (GKX in miniature)

Purged walk-forward: elastic net vs LightGBM on **identical inputs**. The synthetic DGP plants a nonlinear interaction (momentum × value), so trees *should* win — and we can verify they win **for the right reason** via feature importances.

In [ ]:
# Path shim: make the repo root importable when running from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)


In [ ]:
import yaml
from src.data.synthetic import make_synthetic_panel
from src.utils.stats import rank_normalize_cross_section

cfg = yaml.safe_load(open(ROOT / "configs/config.yaml"))
panel, factors, meta = make_synthetic_panel(cfg, seed=cfg["run"]["seed"])
signal_cols = list(meta.index)
panel = rank_normalize_cross_section(panel, signal_cols)
print(panel["date"].nunique(), "months x", panel["ticker"].nunique(), "names")
meta


In [ ]:
from src.backtest.engine import run_model_backtest, comparison_table
from src.models.models import make_linear, make_lgbm
from src.models.icnet import make_icnet

wcfg, ecfg = cfg['walkforward'], cfg['evaluation']
results = {
    'elasticnet': run_model_backtest(panel, signal_cols,
        lambda: make_linear(cfg['models']['linear']), wcfg, ecfg),
    'lightgbm':  run_model_backtest(panel, signal_cols,
        lambda: make_lgbm(cfg['models']['lgbm']), wcfg, ecfg),
    'icnet':     run_model_backtest(panel, signal_cols,
        lambda: make_icnet(cfg['models']['icnet']), wcfg, ecfg),
}
comparison_table(results).round(3)

Note the turnover column: the tree model trades more, so its **net** edge is smaller than its gross edge. Costs are part of the answer, not a footnote.

In [ ]:
results['lightgbm']['feature_importance'].rename('importance').to_frame().round(3)

In [ ]:
import matplotlib.pyplot as plt
for name, r in results.items():
    r['series']['net'].cumsum().plot(label=f'{name} (net)', lw=1.6, figsize=(9, 4.5))
plt.axhline(0, color='k', lw=0.6); plt.legend()
plt.title('Out-of-sample, net of costs (purged walk-forward)'); plt.show()